# 09 · Fusion, a custom kernel, and precision

Pairs with `docs/04` M10-M12. The custom-kernel section runs on CPU (it's just a
conv), so you get a real, checkable result even without a GPU.

In [ ]:
import torch
from gpulab.learn import inspect as I
HAS_CUDA = torch.cuda.is_available()
dev = torch.device("cuda" if HAS_CUDA else "cpu")
print("device:", dev, "| CUDA:", HAS_CUDA)
if not HAS_CUDA:
    print("No CUDA here - cells run on CPU; the timing/memory numbers are only")
    print("meaningful on your RTX 3060. Run this notebook there for the real story.")

## M10 — torch.compile (fewer, bigger kernels)

In [ ]:
from gpulab.models.cnn1d import CNN1D
import numpy as np
model = CNN1D(3, 120).to(dev)
Xg = torch.tensor(np.random.default_rng(0).standard_normal((1024, 120)).astype("float32")).to(dev)
if HAS_CUDA:
    try:
        cmodel = torch.compile(model)
        _ = cmodel(Xg)                       # warm up (pays compile cost once)
        I.cuda_time(lambda: model(Xg),  iters=50)
        I.cuda_time(lambda: cmodel(Xg), iters=50)
    except Exception as e:
        print("torch.compile issue (Triton on Windows is unofficial):", type(e).__name__)
        print("If it fails, run this cell under WSL2.")
else:
    print("GPU-only. Note: Triton/torch.compile on native Windows is unofficial -> WSL2.")

## M11 — write the preprocessing as a kernel (FIR = conv1d)

Savitzky-Golay smoothing is a **fixed convolution**. Build its coefficients, run it as
`F.conv1d`, and validate against scipy in the interior (edges differ by design).

In [ ]:
import numpy as np
import torch.nn.functional as F
from scipy.signal import savgol_filter, savgol_coeffs

w, p = 13, 2
curves = np.random.default_rng(0).standard_normal((32, 500)).astype("float32")

# scipy reference
ref = savgol_filter(curves, w, p, axis=1)

# as a conv1d: coeffs are the FIR kernel (reverse for convolution vs correlation)
coeffs = savgol_coeffs(w, p).astype("float32")
kernel = torch.tensor(coeffs[::-1].copy()).view(1, 1, w)
x = torch.tensor(curves).unsqueeze(1)                 # (N,1,L)
out = F.conv1d(x, kernel, padding=w // 2).squeeze(1).numpy()

interior = slice(w, -w)
print("max abs diff (interior):", np.abs(out[:, interior] - ref[:, interior]).max())
# Your turn: also express -np.gradient as a finite-difference conv1d kernel. # TODO

## M12 — precision & numerics

In [ ]:
x = torch.randn(4096, 4096, device=dev)
ref64 = x.double().sum()
for dt in (torch.float32, torch.bfloat16, torch.float16):
    err = (x.to(dt).sum().double() - ref64).abs().item()
    print(f"{str(dt):16s} sum abs error vs fp64: {err:.4f}")
print("bf16: same exponent range as fp32, coarser mantissa (rarely overflows).")
print("fp16: narrow range -> needs GradScaler in training; bf16 does not.")

> **Concepts to note** (copy into your own theory notebook):
> - `torch.compile` fuses elementwise chains -> fewer launches (helps launch-bound).
> - Triton on native Windows is unofficial; use WSL2 if compile/kernels misbehave.
> - Savitzky-Golay = fixed FIR filter = conv1d; gradient = finite-difference conv.
> - bf16 vs fp16: range vs precision; fp16 training needs loss scaling, bf16 doesn't.
> - Ampere (3060) has bf16/TF32 but NOT fp8 (that's Ada/Hopper).